<a href="https://colab.research.google.com/github/JefersonAlexander/segmentbraintumor/blob/main/02_preprocesado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf
import kagglehub
import numpy as np
import os
import shutil
import tarfile
from tqdm.notebook import tqdm
import nibabel as nib
import os
import random
import cv2  # Working with images
from google.colab import files

##  Download latest version of dataset

In [ ]:
path = kagglehub.dataset_download("dschettler8845/brats-2021-task1")
print("Path to dataset files:", path)

100%|██████████| 12.3G/12.3G [02:21<00:00, 93.2MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/dschettler8845/brats-2021-task1/versions/1


## Show folders downloaded

In [ ]:
base_path = "/kaggle/input/brats-2021-task1"

print(os.listdir(base_path))

## Data Decompression and Extraction into Directory

In [ ]:
working_dir = "/content/kaggle/working/"
os.makedirs(working_dir, exist_ok=True)

for item in os.listdir(working_dir):
    item_path = os.path.join(working_dir, item)
    if os.path.isfile(item_path) or os.path.islink(item_path):
        os.remove(item_path)
    elif os.path.isdir(item_path):
        shutil.rmtree(item_path)
print("Cleaned working directory.")

tar_path = os.path.join(base_path,"BraTS2021_Training_Data.tar")
extract_path = "/content/kaggle/working/BraTS2021"

if not os.path.exists(extract_path):
    os.makedirs(extract_path)
    print("Extracting tar file... (This takes 2-5 mins)")
    with tarfile.open(tar_path, "r") as tar:

        members = tar.getmembers()
        for member in tqdm(members, desc="Extracting files"):
            tar.extract(member, path=extract_path)
    print("Extraction Complete.")
else:
    print("Dataset already extracted.")

## Data preprocesing

In [ ]:
DATA_DIR = "/content/kaggle/working/BraTS2021/"
PROCESSED_DIR = "/content/kaggle/working/processed_unet"
IMAGES_DIR = os.path.join(PROCESSED_DIR, "images")
MASKS_DIR = os.path.join(PROCESSED_DIR, "masks")

os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(MASKS_DIR, exist_ok=True)

In [ ]:
patients = sorted([p for p in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, p))])

PATIENT_LIMIT = 400

random.seed(42)
selected_patients = sorted(random.sample(patients,PATIENT_LIMIT ))

print("Pacientes seleccionados:", len(selected_patients))
print(selected_patients[:5 ])

print(f"Processing {PATIENT_LIMIT} patients out of {len(patients)}...")

## Procesamiento juntando las 4 modalidades de resonancia

In [ ]:
slice_count = 0

for patient in tqdm(selected_patients, desc="Processing Patients"):
    patient_path = os.path.join(DATA_DIR, patient)


    flair_path = os.path.join(patient_path, f"{patient}_flair.nii.gz")
    t1_path = os.path.join(patient_path, f"{patient}_t1.nii.gz")
    t1ce_path = os.path.join(patient_path, f"{patient}_t1ce.nii.gz")
    t2_path = os.path.join(patient_path, f"{patient}_t2.nii.gz")
    seg_path = os.path.join(patient_path, f"{patient}_seg.nii.gz")

    if (not os.path.exists(flair_path)
       or not os.path.exists(seg_path)
       or not os.path.exists(t1_path)
       or not os.path.exists(t1ce_path)
       or not os.path.exists(t2_path)
       ):
        continue


    flair_vol = nib.load(flair_path).get_fdata()
    t1_vol = nib.load(t1_path).get_fdata()
    t1ce_vol = nib.load(t1ce_path).get_fdata()
    t2_vol = nib.load(t2_path).get_fdata()
    seg_vol = nib.load(seg_path).get_fdata()


    flair_vol = cv2.normalize(flair_vol, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    t1_vol = cv2.normalize(t1_vol, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    t1ce_vol = cv2.normalize(t1ce_vol, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    t2_vol = cv2.normalize(t2_vol, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)


    for i in range(30, 120):
        mask_slice = seg_vol[:, :, i]


        if np.max(mask_slice) > 0:
            img_slice = flair_vol[:, :, i]
            t1_slice = t1_vol[:, :, i]
            t1ce_slice = t1ce_vol[:, :, i]
            t2_slice = t2_vol[:, :, i]


            mask_multiclass = mask_slice.astype(np.uint8) #  It conserve the classes


            flair_resized = cv2.resize(img_slice, (128, 128))
            t1_resized = cv2.resize(t1_slice, (128, 128))
            t1ce_resized = cv2.resize(t1ce_slice, (128, 128))
            t2_resized = cv2.resize(t2_slice, (128, 128))

            img_4ch = np.stack(
                [flair_resized, t1_resized, t1ce_resized, t2_resized],
                axis=-1
            )


            mask_resized = cv2.resize(mask_multiclass, (128, 128), interpolation=cv2.INTER_NEAREST)


            filename = f"{patient}_slice_{i}.npy"
            np.save(os.path.join(IMAGES_DIR, filename), img_4ch)

            np.save(os.path.join(MASKS_DIR, filename), mask_resized)

            slice_count += 1
print(f"Preprocessing Done. Created {slice_count} slices.")

## Save data

In [ ]:
shutil.make_archive(
    '/content/kaggle/working/processed_unet',
    'zip',
    '/content/kaggle/working/processed_unet'
)

In [ ]:
zip_path = '/content/kaggle/working/processed_unet.zip'

size_bytes = os.path.getsize(zip_path)

size_gb = size_bytes / (1024**3)

print(f"ZIP size: {size_gb:.2f} GB")

In [ ]:
files.download('/content/kaggle/working/processed_unet.zip')